# Driving Perception Lab — préparation finale
Ce notebook ajoute les checkpoints ImageNet et Driving-Aware I-JEPA, les compacte, les découpe pour GitHub, vérifie leur intégrité et produit le ZIP final. Exécutez les cellules dans l'ordre.

In [ ]:
# 1 — Importer le ZIP source reçu
from google.colab import files
from pathlib import Path
import shutil, subprocess, sys

uploaded = files.upload()
archives = [name for name in uploaded if name.lower().endswith('.zip')]
if len(archives) != 1:
    raise RuntimeError('Importez exactement un fichier ZIP du projet.')

workspace = Path('/content/driving_perception_build')
if workspace.exists():
    shutil.rmtree(workspace)
workspace.mkdir(parents=True)
shutil.unpack_archive(archives[0], workspace)
candidates = [path.parent for path in workspace.rglob('app.py') if (path.parent / 'model.py').is_file()]
if len(candidates) != 1:
    raise RuntimeError(f'Racine du projet ambiguë : {candidates}')
project = candidates[0]
print('Projet détecté :', project)
print('Python Colab :', sys.version.split()[0])
print('Installation de timm, nécessaire à la validation ImageNet…')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'timm==1.0.27'], check=True)
print('Environnement Colab prêt. Python 3.12 sera utilisé uniquement sur Streamlit Cloud.')

In [ ]:
# 2 — Monter Google Drive et sélectionner les deux checkpoints exacts
from google.colab import drive
drive.mount('/content/drive')

IJEPA_CHECKPOINT = Path('/content/drive/MyDrive/PFE_JEPA/NB6_FINAL_COLAB/detection_E2_architecture/checkpoints/best_detector.pt')
IMAGENET_CHECKPOINT = Path('/content/drive/MyDrive/PFE_JEPA/NB2_imagenet_baseline/checkpoints/best_detector.pt')
missing = [path for path in (IJEPA_CHECKPOINT, IMAGENET_CHECKPOINT) if not path.is_file()]
if missing:
    search_root = Path('/content/drive/MyDrive/PFE_JEPA')
    found = list(search_root.rglob('best_detector.pt')) if search_root.exists() else []
    print('Checkpoints trouvés :')
    for item in found:
        print(' -', item)
    raise FileNotFoundError(f'Checkpoint(s) absent(s) : {missing}. Corrigez les chemins avec la liste affichée.')
print('Driving-Aware I-JEPA :', IJEPA_CHECKPOINT)
print('ImageNet baseline     :', IMAGENET_CHECKPOINT)

In [ ]:
# 3 — Compacter, découper et créer le manifeste SHA-256
model_sources = {'ijepa': IJEPA_CHECKPOINT, 'imagenet': IMAGENET_CHECKPOINT}
for model_id, source in model_sources.items():
    model_dir = project / 'models' / model_id
    destination = model_dir / 'best_detector.pt'
    subprocess.run([sys.executable, str(project / 'prepare_checkpoint.py'), str(source), str(destination)], check=True)
    parts = sorted(model_dir.glob('best_detector.pt.part*'))
    manifest = model_dir / 'checkpoint_manifest.json'
    if not parts or not manifest.is_file():
        raise RuntimeError(f'Préparation incomplète pour {model_id}.')
    print(f'{model_id}: {len(parts)} morceaux')
    for part in parts:
        print(f' - {part.name}: {part.stat().st_size / 1024**2:.1f} MiB')

In [ ]:
# 4 — Validation stricte de l'architecture et du checkpoint
subprocess.run([sys.executable, str(project / 'validate_deployment.py')], cwd=project, check=True)

In [ ]:
# 5 — Nettoyer, créer et télécharger le ZIP final
for cache in project.rglob('__pycache__'):
    shutil.rmtree(cache)
for artifact in list(project.glob('*.zip')):
    artifact.unlink()
final_base = Path('/content/DRIVING_PERCEPTION_LAB_FINAL')
final_zip = Path(shutil.make_archive(str(final_base), 'zip', root_dir=project))
print(f'ZIP final validé : {final_zip} ({final_zip.stat().st_size / 1024**2:.1f} MiB)')
files.download(str(final_zip))

## Étape suivante
Décompressez `DRIVING_PERCEPTION_LAB_FINAL.zip`. Sur GitHub, envoyez **son contenu décompressé**, jamais le ZIP lui-même. Lors du déploiement Streamlit, choisissez `app.py` et **Python 3.12** dans Advanced settings.